# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [2]:
from collections import Counter
from pathlib import Path
import sys

import nltk
import numpy as np
import pandas as pd
from IPython.display import display
from nltk.corpus import gutenberg
from nltk.tokenize import RegexpTokenizer
from scipy.sparse import csr_matrix

root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
ruta_nltk = str(root_dir / "data" / "nltk_data")
if ruta_nltk not in nltk.data.path:
    nltk.data.path.insert(0, ruta_nltk)

tokenizador = RegexpTokenizer(r"[A-Za-z]+(?:['’][A-Za-z]+)?")


def tokenizar(texto):
    return tokenizador.tokenize(texto.lower())


try:
    documentos = sorted(gutenberg.fileids())
except LookupError as error:
    raise LookupError(
        "Falta el corpus Gutenberg de NLTK. Descárgalo en data/nltk_data "
        "con nltk.download('gutenberg', download_dir=str(root_dir / 'data/nltk_data'))."
    ) from error

textos_originales = {doc: gutenberg.raw(doc) for doc in documentos}
corpus_procesado = {doc: tokenizar(textos_originales[doc]) for doc in documentos}
# Se conserva el encabezado original como título, incluido autor/año si constan.
titulos = {
    doc: textos_originales[doc].splitlines()[0].strip().strip("[]")
    for doc in documentos
}
longitudes = np.array([len(corpus_procesado[doc]) for doc in documentos])
vocabulario = sorted({t for tokens in corpus_procesado.values() for t in tokens})

resumen_corpus = pd.DataFrame({
    "Identificador": documentos,
    "Título": [titulos[doc] for doc in documentos],
    "Términos únicos": [len(set(corpus_procesado[doc])) for doc in documentos],
    "Longitud (tokens)": longitudes,
})
display(resumen_corpus)
print("Python del kernel:", sys.executable)
print("Documentos:", len(documentos))
print("Tamaño del vocabulario:", len(vocabulario))
print(
    f"Longitud en tokens: mínima={longitudes.min():,}; "
    f"media={longitudes.mean():,.2f}; máxima={longitudes.max():,}"
)

,Identificador,Título,Términos únicos,Longitud (tokens)
0,austen-emma.txt,Emma by Jane Austen 1816,7204,160994
1,austen-persuasion.txt,Persuasion by Jane Austen 1818,5824,83616
2,austen-sense.txt,Sense and Sensibility by Jane Austen 1811,6378,119912
3,bible-kjv.txt,The King James Bible,12789,790052
4,blake-poems.txt,Poems by William Blake 1789,1542,6837
5,bryant-stories.txt,Stories to Tell to Children by Sara Cone Bryan...,3913,46319
6,burgess-busterbrown.txt,The Adventures of Buster Bear by Thornton W. B...,1543,15963
7,carroll-alice.txt,Alice's Adventures in Wonderland by Lewis Carr...,2629,26687
8,chesterton-ball.txt,The Ball and The Cross by G.K. Chesterton 1909,8285,82172
9,chesterton-brown.txt,The Wisdom of Father Brown by G. K. Chesterton...,7882,72389


Python del kernel: c:\Users\johan\Desktop\UNI\7 Semestre\Recuperación_de_Información\.venv\Scripts\python.exe
Documentos: 18
Tamaño del vocabulario: 43495
Longitud en tokens: mínima=6,837; media=117,718.72; máxima=790,052


**Configuración utilizada.** Se retoman los 18 libros de NLTK Gutenberg de la Parte A del notebook 02. La longitud se mide en tokens procesados, no en caracteres ni términos únicos. Se conserva la expresión regular anterior: letras ASCII y apóstrofos internos, sin eliminar palabras vacías, sin stemming ni lematización. Los guiones separan palabras; los dígitos se descartan y las letras acentuadas no se conservan como tales. Se mantienen los encabezados originales para reproducir el corpus anterior. Las consultas estarán en inglés, como los documentos.

### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [3]:
def calcular_tf(tokens):
    """TF absoluto: apariciones de cada término en un documento."""
    return Counter(tokens)


def calcular_df(conteos_documentos):
    """Cada documento aporta como máximo una unidad por término."""
    df = Counter()
    for conteo in conteos_documentos:
        df.update(conteo.keys())
    return df


def calcular_idf(frecuencias_df, n_documentos):
    df = np.asarray(frecuencias_df, dtype=float)
    if n_documentos < 1 or not np.isfinite(df).all() or np.any((df < 1) | (df > n_documentos)):
        raise ValueError("Se requiere N >= 1 y 1 <= DF <= N.")
    return np.log(n_documentos / df) + 1.0


def construir_matriz_tf(conteos_documentos, vocabulario):
    """Construye TF dispersa e ignora términos fuera del vocabulario fijado."""
    posicion = {termino: j for j, termino in enumerate(vocabulario)}
    filas, columnas, valores = [], [], []
    for i, conteo in enumerate(conteos_documentos):
        for termino, frecuencia in conteo.items():
            if termino in posicion:
                filas.append(i)
                columnas.append(posicion[termino])
                valores.append(frecuencia)
    return csr_matrix(
        (valores, (filas, columnas)),
        shape=(len(conteos_documentos), len(vocabulario)),
        dtype=float,
    )


def calcular_tfidf(matriz_tf, vector_idf):
    return matriz_tf.multiply(vector_idf).tocsr()


conteos_ejemplo = [calcular_tf(["a", "a", "b"]), calcular_tf(["a", "c"])]
vocabulario_ejemplo = ["a", "b", "c"]
df_ejemplo = calcular_df(conteos_ejemplo)
idf_ejemplo = calcular_idf([df_ejemplo[t] for t in vocabulario_ejemplo], 2)
tf_ejemplo = construir_matriz_tf(conteos_ejemplo, vocabulario_ejemplo)
pesos_ejemplo = calcular_tfidf(tf_ejemplo, idf_ejemplo)
np.testing.assert_allclose(
    pesos_ejemplo.toarray(),
    [[2, np.log(2) + 1, 0], [1, 0, np.log(2) + 1]],
)
display(pd.DataFrame(
    pesos_ejemplo.toarray(), index=["d1", "d2"], columns=vocabulario_ejemplo
))

,a,b,c
d1,2.0,1.693147,0.000000
d2,1.0,0.000000,1.693147


**Interpretación de los pesos.** La consigna define una sola variante de TF-IDF. Se muestran los seis pesos de un ejemplo de dos documentos por tres términos y luego seis pesos del corpus; no se inventan seis fórmulas distintas. Si TF es cero, el peso es cero. Si un término aparece en todos los documentos, su IDF es 1. Se usa logaritmo natural, a diferencia del `log10(N/DF)` del notebook `tf_idf.ipynb`.

### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [5]:
N = len(documentos)
conteos = [calcular_tf(corpus_procesado[doc]) for doc in documentos]
posicion = {termino: j for j, termino in enumerate(vocabulario)}
frecuencias_df = calcular_df(conteos)
vector_df = np.array([frecuencias_df[t] for t in vocabulario])
vector_idf = calcular_idf(vector_df, N)
matriz_tf = construir_matriz_tf(conteos, vocabulario)
matriz_tfidf = calcular_tfidf(matriz_tf, vector_idf)

# DF se contrasta con la cantidad de filas no nulas de cada columna.
np.testing.assert_array_equal(vector_df, np.asarray((matriz_tf > 0).sum(axis=0)).ravel())
np.testing.assert_array_equal(np.asarray(matriz_tf.sum(axis=1)).ravel(), longitudes)
assert np.isfinite(matriz_tfidf.data).all()

print("Dimensiones documento-término:", matriz_tfidf.shape)
print("Pesos no nulos:", matriz_tfidf.nnz)
display(pd.DataFrame(
    matriz_tfidf[:6, :10].toarray(),
    index=documentos[:6], columns=vocabulario[:10],
).round(6))

# Seis pesos del corpus
ejemplos_pesos = []
for doc in ["austen-emma.txt", "melville-moby_dick.txt"]:
    i = documentos.index(doc)
    for termino in ["whale", "love", "god"]:
        j = posicion[termino]
        ejemplos_pesos.append({
            "Documento": doc, "Término": termino,
            "TF": int(matriz_tf[i, j]), "DF": vector_df[j],
            "IDF = ln(N/DF) + 1": vector_idf[j],
            "TF-IDF": matriz_tfidf[i, j],
        })
display(pd.DataFrame(ejemplos_pesos).round(6))

Dimensiones documento-término: (18, 43495)
Pesos no nulos: 123672


,a,a'lee,a'low,a'm,a'most,a'n,a'ready,a'shiver,a'th,a'top
austen-emma.txt,3130.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
austen-persuasion.txt,1594.0,0.0,0.0,0.0,0.0,3.197225,0.0,0.0,0.0,0.0
austen-sense.txt,2092.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
bible-kjv.txt,8177.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
blake-poems.txt,128.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
bryant-stories.txt,1036.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


,Documento,Término,TF,DF,IDF = ln(N/DF) + 1,TF-IDF
0,austen-emma.txt,whale,0,6,2.098612,0.000000
1,austen-emma.txt,love,117,15,1.182322,138.331622
2,austen-emma.txt,god,8,16,1.117783,8.942264
3,melville-moby_dick.txt,whale,1101,6,2.098612,2310.572130
4,melville-moby_dick.txt,love,24,15,1.182322,28.375717
5,melville-moby_dick.txt,god,136,16,1.117783,152.018493


Las matrices TF y TF-IDF usan formato disperso CSR: solo almacenan los valores no nulos. Los conteos y el DF se calculan explícitamente; SciPy se utiliza para almacenar y operar con las matrices. El vocabulario está ordenado alfabéticamente y se conserva ese orden en todas las representaciones.

### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Tolerancia absoluta para pesos
TOL = 1e-10


def mismos_tokens(tokens):
    return tokens


vectorizador = TfidfVectorizer(
    analyzer=mismos_tokens,
    vocabulary=posicion,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
    dtype=np.float64,
)
matriz_tfidf_sklearn = vectorizador.fit_transform(
    [corpus_procesado[doc] for doc in documentos]
)
np.testing.assert_array_equal(vectorizador.get_feature_names_out(), vocabulario)
np.testing.assert_allclose(vector_idf, vectorizador.idf_, rtol=0, atol=TOL)
diferencia = matriz_tfidf - matriz_tfidf_sklearn
diferencia_maxima = float(np.max(np.abs(diferencia.data), initial=0.0))
assert diferencia_maxima <= TOL
print(f"Máxima diferencia absoluta TF-IDF: {diferencia_maxima:.12g}")
print(f"Coinciden las {N} filas y las {len(vocabulario)} columnas (atol={TOL:g}).")

Máxima diferencia absoluta TF-IDF: 0
Coinciden las 18 filas y las 43495 columnas (atol=1e-10).


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

consultas = pd.DataFrame([
    {
        "Consulta": "whale sea ship ocean",
        "Necesidad": "Relatos de ballenas y navegación marítima.",
    },
    {
        "Consulta": "love marriage family",
        "Necesidad": "Historias de amor, matrimonio y relaciones familiares.",
    },
    {
        "Consulta": "god heaven earth",
        "Necesidad": "Textos religiosos sobre Dios, el cielo y la creación.",
    },
    {
        "Consulta": "murder king ghost",
        "Necesidad": "Tragedias con asesinatos, reyes y apariciones.",
    },
])
tokens_consultas = [tokenizar(q) for q in consultas["Consulta"]]
consultas["Tokens"] = tokens_consultas
consultas["Fuera del vocabulario"] = [
    sorted({t for t in tokens if t not in posicion}) for tokens in tokens_consultas
]
display(consultas)

idf_antes = vectorizador.idf_.copy()
consultas_tfidf_sklearn = vectorizador.transform(tokens_consultas)
consultas_tf = construir_matriz_tf(
    [calcular_tf(tokens) for tokens in tokens_consultas], vocabulario
)
consultas_tfidf = calcular_tfidf(consultas_tf, vector_idf)
diferencia_consultas = consultas_tfidf - consultas_tfidf_sklearn
assert np.max(np.abs(diferencia_consultas.data), initial=0.0) <= TOL
np.testing.assert_array_equal(vectorizador.idf_, idf_antes)

,Consulta,Necesidad,Tokens,Fuera del vocabulario
0,whale sea ship ocean,Relatos de ballenas y navegación marítima.,"[whale, sea, ship, ocean]",[]
1,love marriage family,"Historias de amor, matrimonio y relaciones fam...","[love, marriage, family]",[]
2,god heaven earth,"Textos religiosos sobre Dios, el cielo y la cr...","[god, heaven, earth]",[]
3,murder king ghost,"Tragedias con asesinatos, reyes y apariciones.","[murder, king, ghost]",[]


In [8]:
def coseno_explicito(consultas_pesos, documentos_pesos):
    productos = (consultas_pesos @ documentos_pesos.T).toarray()
    normas_q = np.sqrt(np.asarray(consultas_pesos.multiply(consultas_pesos).sum(axis=1)).ravel())
    normas_d = np.sqrt(np.asarray(documentos_pesos.multiply(documentos_pesos).sum(axis=1)).ravel())
    denominadores = normas_q[:, None] * normas_d[None, :]
    return np.divide(
        productos, denominadores,
        out=np.zeros_like(productos), where=denominadores > 0,
    )


similitudes_manual = coseno_explicito(consultas_tfidf, matriz_tfidf)
similitudes_sklearn = cosine_similarity(consultas_tfidf_sklearn, matriz_tfidf_sklearn)
np.testing.assert_allclose(similitudes_manual, similitudes_sklearn, rtol=0, atol=TOL)
assert np.all((similitudes_manual >= -TOL) & (similitudes_manual <= 1 + TOL))
print("Máxima diferencia de cosenos:", np.max(np.abs(similitudes_manual - similitudes_sklearn)))

Máxima diferencia de cosenos: 2.1094237467877974e-15


In [10]:
i_q = 0
i_d = int(np.argmax(similitudes_manual[i_q]))
q = consultas_tfidf.getrow(i_q)
d = matriz_tfidf.getrow(i_d)
compartidos = np.intersect1d(q.indices, d.indices)
pesos_q = q[:, compartidos].toarray().ravel()
pesos_d = d[:, compartidos].toarray().ravel()
productos_parciales = pesos_q * pesos_d
producto_punto = float((q @ d.T)[0, 0])
norma_q = float(np.sqrt(q.multiply(q).sum()))
norma_d = float(np.sqrt(d.multiply(d).sum()))
np.testing.assert_allclose(productos_parciales.sum(), producto_punto, rtol=0, atol=TOL)

print("Consulta:", consultas.loc[i_q, "Consulta"])
print("Documento:", documentos[i_d])
display(pd.DataFrame({
    "Término compartido": [vocabulario[j] for j in compartidos],
    "TF consulta": [int(consultas_tf[i_q, j]) for j in compartidos],
    "TF documento": [int(matriz_tf[i_d, j]) for j in compartidos],
    "IDF": vector_idf[compartidos],
    "Peso consulta": pesos_q, "Peso documento": pesos_d,
    "Producto parcial": productos_parciales,
}).round(8))
display(pd.DataFrame([{
    "Producto punto": producto_punto,
    "Norma consulta": norma_q, "Norma documento": norma_d,
    "Coseno explícito": producto_punto / (norma_q * norma_d),
    "Coseno sklearn": similitudes_sklearn[i_q, i_d],
}]).round(10))

# Casos límite: consulta vacía, solo signos y término inexistente.
consultas_limite = [tokenizar(q) for q in ["", "123 !!!", "zzqxvnotincorpus"]]
assert all(t not in posicion for t in consultas_limite[-1])
pesos_limite = vectorizador.transform(consultas_limite)
np.testing.assert_allclose(coseno_explicito(pesos_limite, matriz_tfidf), 0, rtol=0, atol=TOL)
np.testing.assert_allclose(
    cosine_similarity(pesos_limite, matriz_tfidf_sklearn), 0, rtol=0, atol=TOL
)

Consulta: whale sea ship ocean
Documento: melville-moby_dick.txt


,Término compartido,TF consulta,TF documento,IDF,Peso consulta,Peso documento,Producto parcial
0,ocean,1,76,1.693147,1.693147,128.679186,217.872801
1,sea,1,454,1.117783,1.117783,507.473498,567.245267
2,ship,1,438,1.325422,1.325422,580.535011,769.454108
3,whale,1,1101,2.098612,2.098612,2310.572130,4848.995066


,Producto punto,Norma consulta,Norma documento,Coseno explícito,Coseno sklearn
0,6403.567242,3.205792,21338.639445,0.093609,0.093609


**Lectura del cálculo.** El producto punto suma únicamente los productos de términos compartidos, pero cada norma se calcula sobre **todo** el vector, incluidos los términos que no comparte con el otro. Las consultas reutilizan los IDF y el vocabulario ajustados al corpus. Las palabras desconocidas se ignoran sin incorporar documentos nuevos.

Cuando una norma es cero, el coseno matemático no está definido. Para recuperar documentos se adopta la convención de scikit-learn: devolver cero. Se verifica este comportamiento con consultas vacías, solo signos y un término fuera del vocabulario. La tolerancia es absoluta (`1e-10`); las comprobaciones se realizan antes de redondear las tablas.

### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [11]:
def ordenar_documentos(puntuaciones):
    return np.lexsort((np.array(documentos), -np.asarray(puntuaciones)))


def obtener_fragmento(doc_id, tokens_consulta, ancho=220):
    texto = textos_originales[doc_id]
    conocidos = {t for t in tokens_consulta if t in posicion}
    inicio_token = next(
        (a for a, b in tokenizador.span_tokenize(texto)
         if texto[a:b].lower() in conocidos),
        None,
    )
    inicio = max(0, inicio_token - 70) if inicio_token is not None else 0
    fin = min(len(texto), inicio + ancho)
    fragmento = " ".join(texto[inicio:fin].split())
    return ("…" if inicio else "") + fragmento + ("…" if fin < len(texto) else "")


def construir_ranking(puntuaciones, tokens_consulta, k=5):
    orden = ordenar_documentos(puntuaciones)[:k]
    return pd.DataFrame([
        {
            "Posición": rango,
            "Identificador": documentos[i],
            "Título": titulos[documentos[i]],
            "Puntuación": float(puntuaciones[i]),
            "Fragmento": obtener_fragmento(documentos[i], tokens_consulta),
        }
        for rango, i in enumerate(orden, start=1)
    ])


rankings_tfidf = []
for i_q, consulta in enumerate(consultas["Consulta"]):
    np.testing.assert_allclose(
        similitudes_manual[i_q], similitudes_sklearn[i_q], rtol=0, atol=TOL
    )
    np.testing.assert_array_equal(
        ordenar_documentos(similitudes_manual[i_q]),
        ordenar_documentos(similitudes_sklearn[i_q]),
    )
    ranking = construir_ranking(similitudes_manual[i_q], tokens_consultas[i_q])
    rankings_tfidf.append(ranking)
    print(f"Consulta {i_q + 1}: {consulta}")
    with pd.option_context("display.max_colwidth", None):
        display(ranking.set_index("Posición").round({"Puntuación": 8}))

Consulta 1: whale sea ship ocean


,Identificador,Título,Puntuación,Fragmento
Posición,,,,
1,melville-moby_dick.txt,Moby Dick by Herman Melville 1851,0.093609,"…you take in hand to school others, and to teach them by what name a whale-fish is to be called in our tongue leaving out, through ignorance, the letter H, which almost alone maketh the signification of the word, you…"
2,whitman-leaves.txt,Leaves of Grass by Walt Whitman 1855,0.011061,"…attles, I above all promote brave soldiers. } In Cabin'd Ships at Sea In cabin'd ships at sea, The boundless blue on every side expanding, With whistling winds and music of the waves, the large imperious waves, Or s…"
3,bryant-stories.txt,Stories to Tell to Children by Sara Cone Bryant 1918,0.006989,"…THE CLOUD One hot summer morning a little Cloud rose out of the sea and floated lightly and happily across the blue sky. Far below lay the earth, brown, dry, and desolate, from drought. The little Cloud could see…"
4,chesterton-ball.txt,The Ball and The Cross by G.K. Chesterton 1909,0.005616,"…K. Chesterton 1909] I. A DISCUSSION SOMEWHAT IN THE AIR The flying ship of Professor Lucifer sang through the skies like a silver arrow; the bleak white steel of it, gleaming in the bleak blue emptiness of the evening…"
5,milton-paradise.txt,Paradise Lost by John Milton 1667,0.005551,"…e, Briareos or Typhon, whom the den By ancient Tarsus held, or that sea-beast Leviathan, which God of all his works Created hugest that swim th' ocean-stream. Him, haply slumbering on the Norway foam, The pilot of…"


Consulta 2: love marriage family


,Identificador,Título,Puntuación,Fragmento
Posición,,,,
1,blake-poems.txt,Poems by William Blake 1789,0.026316,"…ut on earth a little space, That we may learn to bear the beams of love And these black bodies and this sunburnt face Is but a cloud, and like a shady grove. ""For when our souls have learn'd the heat to bear,…"
2,austen-persuasion.txt,Persuasion by Jane Austen 1818,0.013939,"…ter had improved it by adding, for the information of himself and his family, these words, after the date of Mary's birth-- ""Married, December 16, 1810, Charles, son and heir of Charles Musgrove, Esq. of Uppercross, in t…"
3,austen-sense.txt,Sense and Sensibility by Jane Austen 1811,0.013203,"[Sense and Sensibility by Jane Austen 1811] CHAPTER 1 The family of Dashwood had long been settled in Sussex. Their estate was large, and their residence was at Norland Park, in the centre of their property, where, fo…"
4,whitman-leaves.txt,Leaves of Grass by Walt Whitman 1855,0.012894,"…sort to every ship that sails, sail you! Bear forth to them folded my love, (dear mariners, for you I fold it here in every leaf;) Speed on my book! spread your white sails my little bark athwart the imperious wa…"
5,austen-emma.txt,Emma by Jane Austen 1816,0.010720,"…fectionate, indulgent father; and had, in consequence of her sister's marriage, been mistress of his house from a very early period. Her mother had died too long ago for her to have more than an indistinct remembrance o…"


Consulta 3: god heaven earth


,Identificador,Título,Puntuación,Fragmento
Posición,,,,
1,milton-paradise.txt,Paradise Lost by John Milton 1667,0.086136,"…ho first taught the chosen seed In the beginning how the heavens and earth Rose out of Chaos: or, if Sion hill Delight thee more, and Siloa's brook that flowed Fast by the oracle of God, I thence Invoke thy aid to m…"
2,bible-kjv.txt,The King James Bible,0.036622,"…ible The First Book of Moses: Called Genesis 1:1 In the beginning God created the heaven and the earth. 1:2 And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God…"
3,blake-poems.txt,Poems by William Blake 1789,0.024922,"…a child, and thou a lamb, We are called by His name. Little Lamb, God bless thee! Little Lamb, God bless thee! THE LITTLE BLACK BOY My mother bore me in the southern wild, And I am black, but oh my sou…"
4,chesterton-ball.txt,The Ball and The Cross by G.K. Chesterton 1909,0.016613,"…n the bleak blue emptiness of the evening. That it was far above the earth was no expression for it; to the two men in it, it seemed to be far above the stars. The professor had himself invented the flying machine, and…"
5,whitman-leaves.txt,Leaves of Grass by Walt Whitman 1855,0.013885,"…th, Of orbic tendencies to shape and shape and shape, The mighty earth-eidolon. All space, all time, (The stars, the terrible perturbations of the suns, Swelling, collapsing, ending, serving their longer, shor…"


Consulta 4: murder king ghost


,Identificador,Título,Puntuación,Fragmento
Posición,,,,
1,shakespeare-hamlet.txt,The Tragedie of Hamlet by William Shakespeare 1599,0.042292,"…Fran. Nay answer me: Stand & vnfold your selfe Bar. Long liue the King Fran. Barnardo? Bar. He Fran. You come most carefully vpon your houre Bar. 'Tis now strook twelue, get thee to bed Francisco Fr…"
2,shakespeare-macbeth.txt,The Tragedie of Macbeth by William Shakespeare 1603,0.019062,"…gge and filthie ayre. Exeunt. Scena Secunda. Alarum within. Enter King Malcome, Donalbaine, Lenox, with attendants, meeting a bleeding Captaine. King. What bloody man is that? he can report, As seemeth by his plig…"
3,bible-kjv.txt,The King James Bible,0.012676,"[The King James Bible] The Old Testament of the King James Bible The First Book of Moses: Called Genesis 1:1 In the beginning God created the heaven and the earth. 1:2 And the earth was without form, and void; and…"
4,bryant-stories.txt,Stories to Tell to Children by Sara Cone Bryant 1918,0.012671,"…ngle; and in the jungle there was a great big Lion; and the Lion was king of the jungle. Whenever he wanted anything to eat, all he had to do was to come up out of his cave in the stones and earth and _roar_. When he…"
5,carroll-alice.txt,Alice's Adventures in Wonderland by Lewis Carroll 1865,0.011295,"…ought Alice, 'they're sure to kill it in a day or two: wouldn't it be murder to leave it behind?' She said the last words out loud, and the little thing grunted in reply (it had left off sneezing by this time). 'Don't gr…"


El fragmento se toma del texto original alrededor de la primera aparición de un término conocido de la consulta; solo se compactan espacios para mostrarlo. El título y el fragmento no se añaden como campos nuevos al modelo: se utiliza el texto original completo, como en la práctica 02.

Los empates exactos se resuelven por identificador ascendente. Un cero indica ausencia de coincidencia léxica, aunque el documento aparezca al completar cinco posiciones. La igualdad de pesos y cosenos verifica la implementación; sin juicios de relevancia no demuestra la calidad de los resultados.

### 7. Complemento: BM25 y comparación de rankings

Se añade BM25 para cubrir el objetivo inicial del notebook. Se mantienen los mismos documentos, tokens, conteos TF, DF y longitudes; **su IDF es diferente del de TF-IDF**. Se utiliza la variante positiva:

$$
IDF_{BM25}(t)=\ln\left(1+\frac{N-DF(t)+0.5}{DF(t)+0.5}\right)
$$

$$
BM25(q,d)=\sum_{t\in\operatorname{únicos}(q)} IDF_{BM25}(t)
\frac{f(t,d)(k_1+1)}{f(t,d)+k_1\left(1-b+b\frac{|d|}{avgdl}\right)}.
$$

Aquí $|d|$ es la longitud procesada del documento y $avgdl$ la media del corpus. Se fijan $k_1=1.2$ y $b=0.75$, sin ajuste experimental: $k_1$ controla la saturación de TF y $b$ la corrección por longitud. Con $b=0$ se elimina esa corrección. Los términos de consulta repetidos cuentan una sola vez en esta versión; las palabras fuera del corpus no aportan puntuación.

La [documentación de BM25Similarity de Apache Lucene](https://lucene.apache.org/core/7_4_0/core/org/apache/lucene/search/similarities/BM25Similarity.html) recoge esta variante de IDF y esos parámetros. La estructura de saturación se describe en [Introduction to Information Retrieval, sección de Okapi BM25](https://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html). La implementación siguiente evalúa la fórmula declarada, sin intentar reproducir las aproximaciones internas de un motor de búsqueda.

In [9]:
def puntuar_bm25(tokens_consulta, conteos_documentos, df, k1=1.2, b=0.75):
    if not np.isfinite(k1) or k1 <= 0 or not np.isfinite(b) or not 0 <= b <= 1:
        raise ValueError("Se requiere k1 > 0 finito y 0 <= b <= 1.")
    n = len(conteos_documentos)
    longitudes_bm25 = np.array([sum(c.values()) for c in conteos_documentos], dtype=float)
    if n == 0 or not np.any(longitudes_bm25 > 0):
        raise ValueError("El corpus debe contener al menos un documento con tokens.")
    avgdl = longitudes_bm25.mean()
    correccion = k1 * (1 - b + b * longitudes_bm25 / avgdl)
    puntuaciones = np.zeros(n)
    for termino in sorted(set(tokens_consulta)):
        df_t = df.get(termino, 0)
        if df_t == 0:
            continue
        idf_t = np.log1p((n - df_t + 0.5) / (df_t + 0.5))
        frecuencias = np.array([c.get(termino, 0) for c in conteos_documentos], dtype=float)
        # Con b=1 un documento vacío tiene denominador cero y aporta cero.
        denominador = frecuencias + correccion
        factor_tf = np.divide(
            frecuencias * (k1 + 1), denominador,
            out=np.zeros(n), where=denominador > 0,
        )
        puntuaciones += idf_t * factor_tf
    return puntuaciones


# Cálculo independiente a mano sobre los dos documentos del punto 2.
esperado_b = np.log(2) * 2.2 / (1 + 1.2 * (0.25 + 0.75 * 3 / 2.5))
np.testing.assert_allclose(
    puntuar_bm25(["b"], conteos_ejemplo, df_ejemplo), [esperado_b, 0],
    rtol=0, atol=TOL,
)
print(f"Prueba manual BM25 para 'b': [{esperado_b:.10f}, 0]")

puntuaciones_bm25 = np.vstack([
    puntuar_bm25(tokens, conteos, frecuencias_df) for tokens in tokens_consultas
])
assert np.isfinite(puntuaciones_bm25).all()
assert np.all(puntuaciones_bm25 >= 0)

rankings_bm25 = []
comparaciones = []
for i_q, consulta in enumerate(consultas["Consulta"]):
    ranking_bm25 = construir_ranking(puntuaciones_bm25[i_q], tokens_consultas[i_q])
    rankings_bm25.append(ranking_bm25)
    ranking_tfidf = rankings_tfidf[i_q]
    ids_tf = set(ranking_tfidf["Identificador"])
    ids_bm = set(ranking_bm25["Identificador"])
    comparaciones.append({
        "Consulta": consulta,
        "Primero TF-IDF": ranking_tfidf.iloc[0]["Identificador"],
        "Primero BM25": ranking_bm25.iloc[0]["Identificador"],
        "Documentos comunes top-5": len(ids_tf & ids_bm),
        "Jaccard top-5": len(ids_tf & ids_bm) / len(ids_tf | ids_bm),
    })
    print(f"Consulta {i_q + 1}: {consulta}")
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame({
            "Posición": range(1, 6),
            "TF-IDF (documento)": ranking_tfidf["Identificador"].to_numpy(),
            "Coseno": ranking_tfidf["Puntuación"].to_numpy(),
            "BM25 (documento)": ranking_bm25["Identificador"].to_numpy(),
            "Puntuación BM25": ranking_bm25["Puntuación"].to_numpy(),
        }).set_index("Posición").round(8))

comparacion_rankings = pd.DataFrame(comparaciones)
display(comparacion_rankings.round(4))

Prueba manual BM25 para 'b': [0.6407242846, 0]
Consulta 1: whale sea ship ocean


,TF-IDF (documento),Coseno,BM25 (documento),Puntuación BM25
Posición,,,,
1,melville-moby_dick.txt,0.093609,melville-moby_dick.txt,4.899939
2,whitman-leaves.txt,0.011061,bryant-stories.txt,4.543125
3,bryant-stories.txt,0.006989,whitman-leaves.txt,4.408703
4,chesterton-ball.txt,0.005616,shakespeare-hamlet.txt,3.746641
5,milton-paradise.txt,0.005551,chesterton-ball.txt,3.692610


Consulta 2: love marriage family


,TF-IDF (documento),Coseno,BM25 (documento),Puntuación BM25
Posición,,,,
1,blake-poems.txt,0.026316,austen-sense.txt,2.078310
2,austen-persuasion.txt,0.013939,austen-persuasion.txt,2.071438
3,austen-sense.txt,0.013203,austen-emma.txt,2.061886
4,whitman-leaves.txt,0.012894,bible-kjv.txt,1.844618
5,austen-emma.txt,0.010720,whitman-leaves.txt,1.844369


Consulta 3: god heaven earth


,TF-IDF (documento),Coseno,BM25 (documento),Puntuación BM25
Posición,,,,
1,milton-paradise.txt,0.086136,milton-paradise.txt,1.239331
2,bible-kjv.txt,0.036622,bible-kjv.txt,1.233446
3,blake-poems.txt,0.024922,chesterton-ball.txt,1.219280
4,chesterton-ball.txt,0.016613,whitman-leaves.txt,1.206204
5,whitman-leaves.txt,0.013885,melville-moby_dick.txt,1.200309


Consulta 4: murder king ghost


,TF-IDF (documento),Coseno,BM25 (documento),Puntuación BM25
Posición,,,,
1,shakespeare-hamlet.txt,0.042292,shakespeare-hamlet.txt,3.231196
2,shakespeare-macbeth.txt,0.019062,chesterton-brown.txt,3.134068
3,bible-kjv.txt,0.012676,melville-moby_dick.txt,3.034717
4,bryant-stories.txt,0.012671,chesterton-ball.txt,2.846802
5,carroll-alice.txt,0.011295,bible-kjv.txt,2.782425


,Consulta,Primero TF-IDF,Primero BM25,Documentos comunes top-5,Jaccard top-5
0,whale sea ship ocean,melville-moby_dick.txt,melville-moby_dick.txt,4,0.6667
1,love marriage family,blake-poems.txt,austen-sense.txt,4,0.6667
2,god heaven earth,milton-paradise.txt,milton-paradise.txt,4,0.6667
3,murder king ghost,shakespeare-hamlet.txt,shakespeare-hamlet.txt,2,0.2500


**Interpretación.** TF-IDF con coseno divide el producto punto por las normas completas. BM25 suma contribuciones de los términos consultados, satura su frecuencia y corrige por longitud. Estas diferencias pueden cambiar tanto los documentos seleccionados como su orden.

El Jaccard compara los conjuntos del top-5: $|A\cap B|/|A\cup B|$; no considera el orden. La tabla por posiciones permite ver los cambios de orden. Ni Jaccard ni la coincidencia del primer resultado son medidas de relevancia. Las puntuaciones BM25 no están limitadas al intervalo $[0,1]$ y no se comparan numéricamente con los cosenos. Sin juicios de relevancia y evaluación adicional, no se puede concluir qué modelo recupera mejor.

Se conserva el preprocesamiento del notebook 02, incluidas las palabras vacías. Las comparaciones verifican consistencia entre implementaciones, no una configuración óptima de recuperación. Las comprobaciones usan los datos locales reales; BM25 se contrasta con un ejemplo calculable a mano y no con una biblioteca externa de BM25.

En estas cuatro consultas, los top-5 comparten cuatro documentos en los temas marítimo, familiar y religioso (Jaccard = 2/3), y dos en la consulta de tragedias (Jaccard = 1/4). El primer resultado cambia en «love marriage family»: TF-IDF sitúa primero a *Poems* de Blake y BM25 a *Sense and Sensibility*. Son diferencias observadas en este corpus y estas consultas, sin atribuir superioridad a ninguno de los modelos.